In [1]:
import numpy as np
import scipy as sp
import torch
from torch.autograd.functional import jacobian
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from IPython.display import display, clear_output
from numba import jit

In [62]:
def f_cstr(t, y, Da, beta, B):
    """Time derivatives of the CSTR model."""
    x1, x2 = y
    dx1dt = -x1 + Da * (1-x1) * np.exp(x2)
    dx2dt = -x2 + B * Da * (1-x1) * np.exp(x2) - beta * x2
    return [dx1dt, dx2dt]

def myf_cstr_torch(t, y, pars):
    """Time derivatives of the CSTR model."""
    Da, beta, B = pars
    
    x1, x2 = y
    dx1dt = -x1 + Da * (1-x1) * torch.exp(x2)
    dx2dt = -x2 + B * Da * (1-x1) * torch.exp(x2) - beta * x2
    return torch.stack((dx1dt, dx2dt))
    
def get_pars(Da: np.float=0.33, B: np.float=11., beta: np.float=3.):
    pars = Da, beta, B
    return pars

def myf_torch_expandedODE(t, y, Da, beta, B):
    y = y.flatten()
    x1, x2 = y[0:2]
    V = y[2:6].reshape((2,2))
    P = y[6:12].reshape((2,3))
    
    xinput = torch.stack((x1,x2))
    pinput = torch.stack((Da,beta,B))
    
    ffun_xdev = lambda z: myf_cstr_torch(0, z.flatten(), pinput.flatten())
    ffun_pdev = lambda p: myf_cstr_torch(0, xinput.flatten(), p.flatten())
    
    dydt = myf_cstr_torch(t, xinput, pinput).flatten()

    
    dVdt = (jacobian(ffun_xdev, xinput)@V).flatten()
    dPdt = ((jacobian(ffun_xdev, xinput)@P) +
                        jacobian(ffun_pdev, pinput)).flatten()

    return torch.cat((dydt, dVdt, dPdt))#.flatten()

def myf_expandedODE(t, y, Da, beta, B):
    t = torch.tensor(t)
    y = torch.from_numpy(y)
    Da = torch.tensor(Da, requires_grad=True)
    beta = torch.tensor(beta*10, requires_grad=True)
    B = torch.tensor(B*30, requires_grad=True)
    
    return myf_torch_expandedODE(t, y, Da, beta, B).detach().cpu().numpy()

def myf_integrate_pred(t, y0, pars):
    sol = solve_ivp(myf_expandedODE, y0=y0, t_span=[t[0], t[-1]],
                args=(*pars,), t_eval=t,
                rtol=1e-5, atol=1e-8, dense_output=True)
    x = np.moveaxis(sol.y[0:2,:].reshape((2,1,-1)),-1,0)
    dxdx0 = np.moveaxis(sol.y[2:6,:].reshape((2,2,-1)),-1,0)
    dxdp = np.moveaxis(sol.y[6:12,:].reshape((2,3,-1)),-1,0)
    return x, dxdx0, dxdp

def myf_integrate_true(t, y0, pars, t_skip):
    sol = solve_ivp(f_cstr, y0=y0, t_span=[t[0]-t_skip, t[-1]],
                args=(*pars,), t_eval=t,
                rtol=1e-5, atol=1e-8, dense_output=True)
    x = np.moveaxis(sol.y[0:2,:].reshape((2,1,-1)),-1,0)
    return sol.t, x

def my_cost(x_true, x_pred):
    return 0.5 * (1 / x_true.size) * np.sum((x_pred - x_true) ** 2)

def my_grad(x_true, x_pred, dxdp):
#     print(x_true.shape)
#     assert False
    dif = (x_pred - x_true)
    return  (1 / x_true.size) * np.sum((np.swapaxes((x_pred - x_true),-1,-2)@dxdp), axis=0).flatten()

def my_solver(t, pars0, t_skip, Da, max_steps, lr, beta1, beta2, atol, rtol, patience, lr_mult, max_lr):
    pars = get_pars(Da=Da)
    y = [0.5, 5]
    t_true, x_true = myf_integrate_true(t, y, pars, t_skip)
    
    base_lr = lr
    solved = False
    
    cost = []
    mincost = 1e8
    bad_steps = 0
    
    pars_in = pars0

    x0 = np.array(x_true[0,:,:]).flatten()
    V0 = np.eye(2).flatten()
    P0 = np.zeros((2,3)).flatten()
    
    y_in = np.concatenate((x0, V0, P0)).flatten()
    
    steps = 0
    m = np.zeros(pars_in.shape)
    v = np.zeros(pars_in.shape)
    while not solved:
        steps += 1
        
        x_pred, dxdx0_pred, dxdp_pred = myf_integrate_pred(t_true, y_in, pars_in)
        
#         plt.figure()
#         plt.plot(t_true, x_pred[:,0], label='pred')
#         plt.plot(t_true, x_true[:,0], label='true')
#         plt.legend()

#         assert False
        cost_out = my_cost(x_true, x_pred)
        cost.append(cost_out)
        
        if cost_out > mincost:
            bad_steps += 1
        else:
            mincost = cost_out
            bad_steps = 0
            
        if bad_steps > patience:
            lr = lr * lr_mult
            bad_steps = 0
        
        grad = my_grad(x_true, x_pred, dxdp_pred)
        m = beta1 * m + (1-beta1) * grad
        v = beta2 * v + (1-beta1) * (grad **2)
        mhat = m / (1 - beta1 ** steps)
        vhat = v / (1 - beta2 ** steps)
        update = (lr * mhat) / ((vhat ** 0.5) + 1e-8)
        pars_in = pars_in - update
        
        if max(np.abs(grad)) < 1e-2 and cost_out > atol * 100:
            m = np.zeros(pars_in.shape)
            v = np.zeros(pars_in.shape)
            lr = base_lr
            pars_in += (np.random.random_sample(pars_in.shape) - 0.5) * pars_in / 5
            mincost = 1e8
            bad_steps = 0
        
#         if len(cost) > 2:
#             if np.abs((cost[-1] - cost[-2]) / cost[-2]) < rtol:
#                 print('Converged by relative tolerance')
#                 break
                
        if cost[-1] < atol:
            print('Converged by absolute tolerance')
            solved = True
            break
        
        if steps > max_steps:
            print('Did not converge, exceeded max steps')
            break
        if steps % 10 == 0:
            clear_output(wait=True)
            Da, beta, B = pars_in
            display('Iteration '+str(steps)+' Cost: '+str(cost[-1]) + ' pars: ' + str([Da, beta*10, B*30]) + ' grad: ' + str(grad) + ' lr: ' + str(lr))
#             print('steps: ' + str(steps))
#             print('cost: '+ str(cost[-1]))
#             Da, beta, B = pars_in
#             print('pars: ' + str([Da, beta*10, B*30]))
#             print('grad: ' + str(grad))
#             print('last_dxdp: ' + str(dxdp_pred[-1,:,:]))
    
    return solved, pars_in, cost

In [65]:
tin = np.linspace(0,6,100)
t_skip = 1
Da = 0.33
pars0 = np.array([0.5, 5/10, 5/30])
# pars0 = np.array([0.33, 3./10, 11./5])
# pars0 = np.array([0.33,  3., 10.9])
max_steps = 2000
max_lr = np.array([0.01, 0.01, 0.01]) * 100
lr = np.array([0.01, 0.01, 0.01]) * 10
rtol = 1e-5
atol = 1e-5
beta1 = 0.9
beta2 = 0.999
patience = 50
lr_mult = 0.5

solved, pars_in, cost = my_solver(tin, pars0, t_skip, Da, max_steps, lr, beta1, beta2, atol, rtol, patience, lr_mult, max_lr)
print('Solved: ' + str(solved))
Da, beta, B = pars_in
print('Pars: ' + str([Da, beta*10, B*30]))
print('Final Cost: ' + str(cost[-1]))

'Iteration 2000 Cost: 0.15521927957738282 pars: [0.6748893974692596, 5.425734718907766, 14.087637301627023] grad: [-0.40223636  0.03534756 -0.02674024] lr: [0.1 0.1 0.1]'

Did not converge, exceeded max steps
Solved: False
Pars: [0.6644052518597889, 5.4606153809161455, 14.945609987710014]
Final Cost: 0.15017512367113908


In [ ]:
tin = np.linspace(0,6,100)
Da = 0.33

y0 = [0.5, 5]
pars = get_pars(Da=0.33)
t_skip = 1

t_true, x_true = myf_integrate_true(tin, y0, pars, t_skip)
x, dxdx0, dxdp

print(t_true.shape)
print(x_true.shape)

x0 = np.array([1, 4]).flatten()
V0 = np.eye(2).flatten()
P0 = np.zeros((2,3)).flatten()

y0 = np.concatenate((x0, V0, P0)).flatten()

pars = np.array([0.5, 5., 5.])

x_pred, dxdx0_pred, dxdp_pred = myf_integrate_pred(t_true, y0, pars)
print('=============================')
print(x_pred.shape)
print(dxdx0_pred.shape)
print(dxdp_pred.shape)

In [ ]:
from scipy.integrate import solve_ivp

Da = 0.33
pars = get_pars(Da)

x0 = np.array([0.5, 3]).flatten()
V0 = np.eye(2).flatten()
P0 = np.zeros((2,3)).flatten()

y0 = np.concatenate((x0, V0, P0)).flatten()

sol = solve_ivp(myf_expandedODE, y0=y0, t_span=[0, 6],
                args=(*pars,),
                rtol=1e-5, atol=1e-8, dense_output=True)#, events=(event,))#, dense_output=True)

In [ ]:
print('Final Jacobians')
print(sol.y.shape)
# print(sol.y[:2,-1])
print(sol.y[2:6,-3:])
print(sol.y[2:6,-3:].reshape((2,2,-1)))
print(sol.y[2:6,:].reshape((2,2,-1)).shape)
print('=======================')
print(sol.y[2:6,-1].reshape((2,2)))
print(sol.y[2:6,-3:].reshape((2,2,-1))[:,:,-1])
# print(sol.y[2:6,-1].reshape((2,2)).flatten().reshape((2,2)))
# print(sol.y[6:12,-1].reshape((2,3)))
print('=======================')
x = np.moveaxis(sol.y[0:2,:].reshape((2,1,-1)),-1,0)
dxdx0 = np.moveaxis(sol.y[2:6,:].reshape((2,2,-1)),-1,0)
dxdp = np.moveaxis(sol.y[6:12,:].reshape((2,3,-1)),-1,0)
print(sol.t.shape)
print(x.shape)
print(dxdx0.shape)
print(dxdp.shape)
print(np.matmul(dxdx0, dxdp).shape)
print(np.sum(np.matmul(np.swapaxes(x,-1,-2), dxdp),axis=0).shape)

In [ ]:
x1 = x[:,0]
x2 = x[:,1]

plt.figure()
plt.plot(sol.t, x1)

plt.figure()
plt.plot(sol.t, x2)